# Ses Üretimi (Chatterbox TTS) — Colab GPU

`hasta_cumleleri.jsonl` içindeki, **sesi henüz üretilmemiş** cümleleri Chatterbox
Multilingual TTS ile seslendirir. Mac'te ~9-10 s/cümle idi; Colab GPU'da çok daha
hızlı olması bekleniyor (2. hücrede **kalibrasyon** var — gerçek hızı ölçmeden
kapsam kararı vermeyin).

## Güncel durum

`hasta_cumleleri.jsonl` artık **10.314 satır** (18 dal + `ortak` + `ilac` — sadece
ilaç/acil_tıp değil, tüm branşlar: romatoloji, üroloji, kardiyoloji, dermatoloji,
pediatri, psikiyatri, kbb, göz, ortopedi, gastroenteroloji, nöroloji, endokrinoloji,
kadın_doğum, diş, göğüs_hastalıkları, enfeksiyon, dahiliye, acil_tıp). Hedef artık
**15 değil 12 saat TOPLAM medikal ses** (tüm dallar dahil, sadece ilaç değil).

## DRIVE'A YÜKLEMEN GEREKEN TEK KLASÖR: `sontrain`

Her şey tek bir hazır klasörde: **`sontrain/`**. Bunu olduğu gibi Drive'ın
köküne (My Drive) yükle, böylece Drive'da şu yol oluşsun:

```
Drive/sontrain/veri_uretimi/terimler.py
Drive/sontrain/veri_uretimi/ses_profilleri.py
Drive/sontrain/veri_uretimi/ses_kirp.py
Drive/sontrain/veri_uretimi/kalite_kontrol.py
Drive/sontrain/veri_uretimi/ses_referans/   (48 dosya)
Drive/sontrain/data/hasta_cumleleri.jsonl   (GÜNCEL, 10.314 satır)
Drive/sontrain/data/whisper_manifest.jsonl  (mevcut ses kayıtlarının manifesti)
```

2. hücre bu yolu **ilk sırada** arıyor — başka bir yere yüklersen de (glob ile)
otomatik bulunur, ama en garantisi ve en hızlısı `sontrain`'i doğrudan Drive'ın
köküne koymak.

## Mac'teki `ses_uret.py`'den farkı (ÖNEMLİ)

Orijinal script resume'u **tam dosya yolundan** yapıyor (`/Users/ozanpatlar/...`).
Colab'da yollar farklı olduğu için hiçbir satırı atlamaz ve tüm metni baştan
üretmeye kalkardı. Bu sürüm resume'u **dosya adına göre** yapar — Mac'te zaten
üretilmiş klipler atlanır, sadece eksik olanlar üretilir.

Ayrıca Colab manifesti **ayrı bir dosyaya** (`whisper_manifest_colab.jsonl`) yazar —
Mac'teki manifest bozulmaz, sonra birleştirilir.

## Üretim parametreleri (DEĞİŞTİRİLMEMELİ)

`repetition_penalty=1.2, temperature=0.8, cfg_weight=0.5` — Mac'te canlı test edilip
onaylandı. Varsayılan `repetition_penalty=2.0` kısa cümlelerde token tekrarı döngüsüne
girip sonda "drone" artefaktı üretiyordu.

## Çift kalite kontrolü (korunuyor)

1. **Süre eşiği**: kelime başına en az 0.12s
2. **Whisper geri-transkript**: üretilen sesi Whisper `small` ile okutup hedef metnin
   en az %70'i söylenmiş mi diye bakar. Süre eşiği tek başına yetersizdi — 8 kelimelik
   bir cümle "yeterince uzun" görünüp sadece ilk 3 kelimesini içerebiliyor (gözlemlendi).

Başarısız üretim 3 kez denenir, sonra atlanır.


In [ ]:
# 1) Kurulum
# setuptools<81 SART: perth (watermark) kutuphanesi pkg_resources istiyor,
# setuptools 84+'ta kaldirildi ve hata veriyor.
#
# torchvision ve torchaudio'yu BILEREK ayni pip komutuna dahil ediyoruz: pip'in
# bagimlilik cozucusu boylece chatterbox-tts'in gerektirdigi torch surumuyle
# EZAMANLI uyumlu bir torchvision/torchaudio secer. Bunlari ayri komutlarda
# kurmak (ya da hic belirtmemek) pip'in Colab'in onceden yuklu, artik uyumsuz
# torchvision'ini OLDUGU GIBI birakmasina yol aciyordu - "operator
# torchvision::nms does not exist" hatasi buradan geliyordu. torchvision'i
# tamamen kaldirmak da cozum degil: transformers bazi model importlarinda
# (LlamaModel dahil, Whisper'la alakasiz olsa bile) torchvision'i sart kosuyor.
!pip install -q chatterbox-tts torchvision torchaudio "setuptools<81" faster-whisper silero-vad
!pip uninstall -y torchao

import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2) Kaynaklari OTOMATIK kesfet
import json, os, re, sys, time, random, collections, hashlib, glob
from pathlib import Path

KOK = "/content/drive/MyDrive"

# --- Kod dizini: terimler.py + ses_profilleri.py + ses_referans/ nerede ---
def kod_dizini_bul():
    adaylar = [
        f"{KOK}/sontrain/veri_uretimi",
        f"{KOK}/colab_aktarim/sontrain/veri_uretimi",
        f"{KOK}/catpi_windows_aktarim/veri_uretimi",
        f"{KOK}/colab_aktarim/ses_uretim_paketi/veri_uretimi",
        f"{KOK}/ses_uretim_paketi/veri_uretimi",
    ]
    for a in adaylar:
        if os.path.exists(f"{a}/terimler.py"):
            return a
    for y in glob.glob(f"{KOK}/**/terimler.py", recursive=True):
        return os.path.dirname(y)
    return None

KOD_DIZIN = kod_dizini_bul()
if KOD_DIZIN is None:
    raise SystemExit("terimler.py Drive'da bulunamadi - veri_uretimi/ klasorunu yukle")
print(f"Kod dizini      : {KOD_DIZIN}")
print(f"  ses_referans  : {'VAR' if os.path.isdir(KOD_DIZIN + '/ses_referans') else 'YOK!'}")
print(f"  gemini_prompt : {'VAR' if os.path.exists(KOD_DIZIN + '/gemini_prompt.txt') else 'YOK!'}")

# --- Metin dosyasi: birlesik (tam) varsa onu, yoksa en buyuk hasta_cumleleri ---
METIN_TAM = f"{KOK}/sontrain/data/hasta_cumleleri.jsonl"
if os.path.exists(METIN_TAM) and os.path.getsize(METIN_TAM) > 0:
    METIN_DOSYASI = METIN_TAM
else:
    adaylar = glob.glob(f"{KOK}/**/hasta_cumleleri.jsonl", recursive=True)
    if not adaylar:
        raise SystemExit("hasta_cumleleri.jsonl bulunamadi")
    METIN_DOSYASI = max(adaylar, key=os.path.getsize)
print(f"Metin dosyasi   : {METIN_DOSYASI}")

# --- MANIFESTLER: hepsinin BIRLESIMI ---
# Drive'da hem eski (634 satirlik, Windows aktarimindan) hem guncel (5512) manifest
# duruyor. "Dogrusunu sec" yerine HEPSINI okuyup birlestiriyoruz - boylece eski
# bir manifest zarar vermez, sadece alt kume katkisi yapar. Yanlis manifest secmek
# 4879 ilac klibinin bastan uretilmesine yol acardi.
MANIFESTLER = sorted(set(glob.glob(f"{KOK}/**/whisper_manifest*.jsonl", recursive=True)))
print(f"\nManifestler ({len(MANIFESTLER)} dosya, birlesim alinacak):")

SESLER = {}   # dosya_adi -> (dal, duration_s)
for y in MANIFESTLER:
    n = 0
    for l in open(y, encoding="utf-8"):
        l = l.strip()
        if not l:
            continue
        try:
            r = json.loads(l)
        except json.JSONDecodeError:
            continue
        p, d = r.get("audio_path"), r.get("duration_s")
        if p and d:
            SESLER[os.path.basename(p.replace("\\", "/"))] = (r.get("dal"), d)
            n += 1
    print(f"  {n:>5} satir  {y}")

# --- Diskte fiilen duran WAV'lar: tum dizinlerin BIRLESIMI ---
WAV_DIZINLERI = sorted({os.path.dirname(p) for p in glob.glob(f"{KOK}/**/*.wav", recursive=True)
                        if "/ses_referans" not in p})
DISKTE = set()
print(f"\nWAV dizinleri ({len(WAV_DIZINLERI)}):")
for d in WAV_DIZINLERI:
    w = {f for f in os.listdir(d) if f.endswith(".wav")}
    DISKTE |= w
    print(f"  {len(w):>5} wav   {d}")

print(f"\nBIRLESIM: manifestlerde {len(SESLER)}, diskte {len(DISKTE)}, "
      f"toplam bilinen {len(SESLER | DISKTE if isinstance(SESLER, set) else set(SESLER) | DISKTE)}")

# --- Cikti: egitimin okudugu yer ---
CIKTI_DATA = f"{KOK}/colab_aktarim/data"
SES_CIKTI = Path(f"{CIKTI_DATA}/sesler")
os.makedirs(CIKTI_DATA, exist_ok=True)
SES_CIKTI.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, KOD_DIZIN)
import torch
import torchaudio as ta
from ses_profilleri import profil_sec
from ses_kirp import sesi_kirp
import terimler
print(f"\nModuller yuklendi ({len(terimler.DRUGS_ORIGINAL_ADLAR)} ilac ad eslemesi)")

# --- Uretim ayarlari (Mac'te dogrulanmis, DEGISTIRILMEMELI) ---
HEDEF_SR = 16000
MIN_SANIYE_KELIME_BASI = 0.12
MIN_TRANSKRIPT_ORANI = 0.7
MAX_SES_DENEME = 3

# --- Kapsam ---
DAL_FILTRESI = None    # None = tum dallar
LIMIT = 20             # KALIBRASYON: ilk calistirmada 20, hizi gor, sonra None yap

COLAB_MANIFEST = f"{CIKTI_DATA}/whisper_manifest_colab.jsonl"
print(f"\nSes cikti dizini: {SES_CIKTI}")
print(f"Yeni manifest   : {COLAB_MANIFEST}")


In [ ]:
# 2b) KALICI DUZELTMELER (her runtime baslatmada calistir - restart sonrasi da gerekli)
# NOT: bu hucre kac kez calistirilirsa calistirilsin GUVENLI (idempotent).

# 1) profil_havuzu.json Mac'te uretildigi icin MUTLAK Mac yollari iceriyor
#    (orn. /Users/ozanpatlar/Desktop/catpi/veri_uretimi/ses_referans/...).
#    Colab'da bu yol yok - FileNotFoundError buradan geliyordu. Orijinal
#    fonksiyonu HER ZAMAN ses_profilleri MODULUNUN KENDI attribute'undan aliyoruz
#    (notebook'un global "profil_sec" adindan DEGIL) - bu, hucre birden fazla kez
#    calistirilsa bile kendi kendine referans veren bir dongu (RecursionError)
#    olusmasini engelliyor.
from pathlib import Path as _Path
import ses_profilleri as _sp

if not getattr(_sp, "_yol_duzeltildi", False):
    _orijinal_profil_sec = _sp.profil_sec
    _SES_REFERANS_DIZINI = _Path(KOD_DIZIN) / "ses_referans"

    def _duzeltilmis_profil_sec():
        anahtar, yol = _orijinal_profil_sec()
        return anahtar, str(_SES_REFERANS_DIZINI / _Path(yol).name)

    _sp.profil_sec = _duzeltilmis_profil_sec
    _sp._yol_duzeltildi = True

profil_sec = _sp.profil_sec  # notebook'un global adini da guncelle (from-import ile geldi)
print("profil_sec() Colab icin duzeltildi, test:", profil_sec())

# 2) chatterbox-tts 0.1.3+ (Colab'in kurdugu 0.1.7 dahil) alignment_stream_analyzer.py
#    icinde bug var: art arda SADECE 2 ayni token gorunce (kod yorumu "3x" diyor ama
#    gercek kontrol son 2 tokene bakiyor - github.com/resemble-ai/chatterbox/issues/519)
#    zorla EOS uretip cumleyi cok erken kesiyor. Duzeltme: esigi 2'den 5 ardisik
#    ayni tokene cikariyoruz. (bu kisim zaten idempotent)
from chatterbox.models.t3.inference.alignment_stream_analyzer import AlignmentStreamAnalyzer
import torch as _torch

_REPETITION_ESIGI = 5

def _duzeltilmis_step(self, logits, next_token=None):
    aligned_attn = _torch.stack(self.last_aligned_attns).mean(dim=0)
    i, j = self.text_tokens_slice
    if self.curr_frame_pos == 0:
        A_chunk = aligned_attn[j:, i:j].clone().cpu()
    else:
        A_chunk = aligned_attn[:, i:j].clone().cpu()
    A_chunk[:, self.curr_frame_pos + 1:] = 0
    self.alignment = _torch.cat((self.alignment, A_chunk), dim=0)
    A = self.alignment
    T, S = A.shape

    cur_text_posn = A_chunk[-1].argmax()
    discontinuity = not(-4 < cur_text_posn - self.text_position < 7)
    if not discontinuity:
        self.text_position = cur_text_posn

    false_start = (not self.started) and (A[-2:, -2:].max() > 0.1 or A[:, :4].max() < 0.5)
    self.started = not false_start
    if self.started and self.started_at is None:
        self.started_at = T

    self.complete = self.complete or self.text_position >= S - 3
    if self.complete and self.completed_at is None:
        self.completed_at = T

    long_tail = self.complete and (A[self.completed_at:, -3:].sum(dim=0).max() >= 5)
    alignment_repetition = self.complete and (A[self.completed_at:, :-5].max(dim=1).values.sum() > 5)

    if next_token is not None:
        if isinstance(next_token, _torch.Tensor):
            token_id = next_token.item() if next_token.numel() == 1 else next_token.view(-1)[0].item()
        else:
            token_id = next_token
        self.generated_tokens.append(token_id)
        if len(self.generated_tokens) > 8:
            self.generated_tokens = self.generated_tokens[-8:]

    token_repetition = (
        len(self.generated_tokens) >= _REPETITION_ESIGI and
        len(set(self.generated_tokens[-_REPETITION_ESIGI:])) == 1
    )

    if cur_text_posn < S - 3 and S > 5:
        logits[..., self.eos_idx] = -2**15

    if long_tail or alignment_repetition or token_repetition:
        logits = -(2**15) * _torch.ones_like(logits)
        logits[..., self.eos_idx] = 2**15

    self.curr_frame_pos += 1
    return logits

AlignmentStreamAnalyzer.step = _duzeltilmis_step
print(f"alignment_stream_analyzer.step() duzeltildi - tekrar esigi artik {_REPETITION_ESIGI} ardisik ayni token")


In [ ]:
# 3) Hangi cumlelerin sesi eksik - resume DOSYA ADINA gore (yola gore DEGIL)
def dosya_adi_uret(row):
    """Mac'teki ses_uret.py ile BIREBIR ayni hash - ayni cumle ayni dosya adini uretir."""
    anahtar = f"{row['dal']}|{row['socrates_asama']}|{row['hedef_terim']}|{row['tekrar_no']}|{row.get('kaynak', 'lmstudio')}"
    return f"{row['dal']}_{hashlib.sha1(anahtar.encode('utf-8')).hexdigest()[:12]}.wav"


def manifest_dosya_adlari(*yollar):
    """Manifest(ler)deki audio_path'lerin SADECE DOSYA ADINI toplar - Mac ve Colab
    yollari farkli oldugu icin tam yol karsilastirmasi calismaz."""
    adlar = set()
    for yol in yollar:
        if not yol or not os.path.exists(yol):
            continue
        with open(yol, encoding="utf-8") as f:
            for satir in f:
                satir = satir.strip()
                if not satir:
                    continue
                try:
                    r = json.loads(satir)
                except json.JSONDecodeError:
                    continue
                p = r.get("audio_path")
                if p:
                    adlar.add(os.path.basename(p.replace("\\", "/")))
    return adlar


satirlar = [json.loads(l) for l in open(METIN_DOSYASI, encoding="utf-8") if l.strip()]
if DAL_FILTRESI:
    satirlar = [r for r in satirlar if r.get("dal") == DAL_FILTRESI]

islenmis = set(SESLER)
# Drive'da fiilen duran dosyalari da islenmis say (manifest yazilmadan kopmus olabilir)
diskte = DISKTE
print(f"Manifest(ler)de kayitli: {len(islenmis)}, Drive'da fiilen duran: {len(diskte)}")

islenecekler = []
for row in satirlar:
    ad = dosya_adi_uret(row)
    if ad in islenmis or ad in diskte:
        continue
    islenecekler.append((row, str(SES_CIKTI / ad)))

print(f"\nToplam metin satiri: {len(satirlar)}")
print(f"Sesi zaten var: {len(satirlar) - len(islenecekler)}")
print(f"Sesi EKSIK: {len(islenecekler)}")

import collections
print("Eksiklerin dal dagilimi:", dict(collections.Counter(r["dal"] for r, _ in islenecekler)))

if LIMIT is not None:
    islenecekler = islenecekler[:LIMIT]
    print(f"\nLIMIT={LIMIT} -> bu calistirmada sadece {len(islenecekler)} uretilecek (KALIBRASYON)")

In [ ]:
# 4) Modelleri yukle
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from faster_whisper import WhisperModel

cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {cihaz}")

t0 = time.monotonic()
print("Chatterbox yukleniyor...")
model = ChatterboxMultilingualTTS.from_pretrained(device=cihaz)
print(f"Chatterbox yuklendi: {time.monotonic() - t0:.1f}s")

# Dogrulama Whisper'i: Mac'te CPU'daydi (GPU'yu Chatterbox kullaniyordu). Colab'in
# CPU'su zayif (2 vCPU) - GPU'da calistirmak cok daha hizli, T4'un 15GB'i ikisine yeter.
# base DEGIL small: base'in Turkce'de zayif oldugunu kendi testlerimizde gormustuk,
# hakem rolunde guvenilmez olurdu.
try:
    whisper_dogrulama = WhisperModel("small", device="cuda", compute_type="float16")
    print("Dogrulama Whisper: GPU (float16)")
except Exception as e:
    print(f"GPU'da yuklenemedi ({e}) - CPU'ya dusuluyor")
    whisper_dogrulama = WhisperModel("small", device="cpu", compute_type="int8")
    print("Dogrulama Whisper: CPU (int8)")

In [ ]:
# 5) Uretim dongusu
import numpy as np


def resmi_isme_cevir(cumle, hedef_terim):
    """TTS'e giden metin fonetik yazimi kullanir ("Zanaks") cunku Turkce TTS
    "Xanax" gibi orijinal yazimlari hatali okuyor. Ama EGITIM ETIKETI resmi
    ilac ismini icermeli. Bu fonksiyon sadece etiket metnini duzeltir."""
    resmi = terimler.DRUGS_ORIGINAL_ADLAR.get(hedef_terim)
    if resmi is None or resmi == hedef_terim:
        return cumle
    desen = re.compile(r"\b" + re.escape(hedef_terim) + r"('\w*)?\b")
    return desen.sub(lambda m: resmi + (m.group(1) or ""), cumle)


def icerik_tam_mi(wav16, hedef_metin):
    ses = wav16[0].numpy().astype(np.float32)
    segments, _ = whisper_dogrulama.transcribe(ses, language="tr", beam_size=1, temperature=0.0)
    transkript = " ".join(s.text for s in segments)
    hedef_kelime = len(hedef_metin.split())
    if hedef_kelime == 0:
        return True
    return (len(transkript.split()) / hedef_kelime) >= MIN_TRANSKRIPT_ORANI


uretilen, hatali = 0, 0
t_baslangic = time.monotonic()

with open(COLAB_MANIFEST, "a", encoding="utf-8") as manifest_f:
    for i, (row, yol) in enumerate(islenecekler, 1):
        cumle = row["hasta_cumlesi"]
        min_sure = len(cumle.split()) * MIN_SANIYE_KELIME_BASI
        profil_anahtari, ses_referans_yolu = profil_sec()

        try:
            wav16 = None
            for deneme in range(1, MAX_SES_DENEME + 1):
                wav = model.generate(
                    cumle, language_id="tr", audio_prompt_path=ses_referans_yolu,
                    repetition_penalty=1.2, temperature=0.8, cfg_weight=0.5,
                )
                wav = sesi_kirp(wav, model.sr)
                aday = ta.functional.resample(wav, model.sr, HEDEF_SR)
                aday_sure = aday.shape[-1] / HEDEF_SR
                if aday_sure < min_sure:
                    print(f"  [deneme {deneme}/{MAX_SES_DENEME}] sure cok kisa ({aday_sure:.2f}s < {min_sure:.2f}s): \"{cumle[:40]}...\"")
                    continue
                if not icerik_tam_mi(aday, cumle):
                    print(f"  [deneme {deneme}/{MAX_SES_DENEME}] icerik eksik/kesik: \"{cumle[:40]}...\"")
                    continue
                wav16 = aday
                break

            if wav16 is None:
                print(f"  [hata] {MAX_SES_DENEME} denemede uretilemedi, atlaniyor: \"{cumle[:50]}...\"")
                hatali += 1
                continue

            ta.save(yol, wav16, HEDEF_SR, encoding="PCM_S", bits_per_sample=16)
            sure = wav16.shape[-1] / HEDEF_SR
            etiket = resmi_isme_cevir(cumle, row["hedef_terim"]) if row["dal"] == "ilac" else cumle
            manifest_f.write(json.dumps({
                "audio_path": yol,
                "text": etiket,
                "sample_rate": HEDEF_SR,
                "duration_s": round(sure, 3),
                "dal": row["dal"],
                "socrates_asama": row["socrates_asama"],
                "hedef_terim": row["hedef_terim"],
                "persona": row.get("persona"),
                "kaynak": row.get("kaynak", "lmstudio"),
                "ses_profili": profil_anahtari,
            }, ensure_ascii=False) + "\n")
            manifest_f.flush()
            uretilen += 1
        except Exception as e:
            print(f"  [hata] \"{cumle[:50]}...\": {type(e).__name__}: {e}")
            hatali += 1

        if i % 10 == 0 or i == len(islenecekler):
            gecen = time.monotonic() - t_baslangic
            hiz = gecen / i
            kalan = (len(islenecekler) - i) * hiz
            print(f"  [{i}/{len(islenecekler)}] {hiz:.2f}s/cumle, tahmini kalan: {kalan/60:.1f} dk")

gecen = time.monotonic() - t_baslangic
print(f"\nTamamlandi. Uretilen: {uretilen}, hatali: {hatali}, sure: {gecen/60:.1f} dk")
if uretilen:
    hiz = gecen / (uretilen + hatali)
    kalan_toplam = len(islenecekler) - i  # bu calistirmada henuz islenmemis kalan (LIMIT'ten dolayi)
    print(f"\n=== KALIBRASYON SONUCU ===")
    print(f"Gercek hiz: {hiz:.2f} s/cumle  (Mac/MPS'te ~9.5 s/cumle idi)")
    print(f"1000 cumle -> {1000 * hiz / 3600:.2f} saat")
    print(f"Bu calistirmada toplam eksik olan {len(islenecekler)} cumle -> {len(islenecekler) * hiz / 3600:.2f} saat")
    if kalan_toplam > 0:
        print(f"  (LIMIT={LIMIT} oldugu icin bu calistirma sadece {i} tanesini isledi, {kalan_toplam} tanesi LIMIT'i None yapinca islenecek)")

In [ ]:
# 6) Uretim ozeti - toplam ses suresi ne oldu
def manifest_ozet(*yollar):
    import collections
    sure = collections.Counter(); sayi = collections.Counter(); gorulen = set()
    for yol in yollar:
        if not yol or not os.path.exists(yol):
            continue
        for satir in open(yol, encoding="utf-8"):
            satir = satir.strip()
            if not satir:
                continue
            r = json.loads(satir)
            ad = os.path.basename(r["audio_path"].replace("\\", "/"))
            if ad in gorulen:
                continue
            gorulen.add(ad)
            sure[r["dal"]] += r.get("duration_s", 0)
            sayi[r["dal"]] += 1
    return sayi, sure


sayi, sure = manifest_ozet(*MANIFESTLER, COLAB_MANIFEST)
print("TOPLAM MEDIKAL SES (Mac + Colab, mukerrersiz):")
for dal, n in sayi.most_common():
    print(f"  {dal:<16} {n:>5} klip  {sure[dal]/3600:>5.2f} saat")
print(f"  {'TOPLAM':<16} {sum(sayi.values()):>5} klip  {sum(sure.values())/3600:>5.2f} saat")
if sum(sayi.values()):
    print(f"\nOrtalama klip suresi: {sum(sure.values())/sum(sayi.values()):.2f} s")

# HEDEF: 12 saat TOPLAM medikal ses - artik sadece ilac/acil_tip degil, TUM
# dallar (romatoloji, uroloji, kardiyoloji, ... + ortak + ilac) hedefe dahil.
HEDEF_SAAT = 12.0
med_sn = sum(sure.values())
print(f"\n=== HEDEF DURUMU ===")
print(f"Toplam medikal ses: {med_sn/3600:.2f} / {HEDEF_SAAT} saat  (%{100*med_sn/(HEDEF_SAAT*3600):.1f})")
if med_sn < HEDEF_SAAT * 3600:
    ort = sum(sure.values())/sum(sayi.values())
    print(f"Eksik: {(HEDEF_SAAT*3600-med_sn)/3600:.2f} saat = ~{round((HEDEF_SAAT*3600-med_sn)/ort)} klip daha")
else:
    print("HEDEFE ULASILDI")
